# 📈 02. Modelos de Machine Learning y Pronóstico de Series de Tiempo de Mercado
**Proyecto**: AgroStats AndTech — Plataforma de Inteligencia Agrícola Colombiana  
**Capa del Lakehouse**: **Gold** (Feature Store & DuckDB Star Schema)  
**Técnicas**: SARIMAX con Covariables Exógenas, Suavizamiento Holt-Winters, Random Forest Lagged Regressor  
**Estándares**: DAMA-BOK, ISO/IEC 25010 (Precisión y Fiabilidad), Time-Series Walk-Forward Validation  

---

### Objetivos del Cuaderno:
1. **Descomposición y Estacionariedad**: Descomponer la serie temporal en Tendencia, Estacionalidad y Residuos; verificar con el Test Aumentado de Dickey-Fuller (ADF).
2. **Modelos Econométricos**: Ajustar Holt-Winters y SARIMAX con variables climáticas de precipitación y temperatura como covariables exógenas.
3. **Modelos de Machine Learning**: Entrenar un Random Forest Regressor con rezagos temporales ($t-1, t-7, t-14$) y ventanas móviles de volatilidad.
4. **Evaluación de Desempeño**: Comparar $MAE, RMSE, MAPE$ y generar curvas de pronóstico a 14 días con intervalos de confianza al 95%.


In [1]:
import matplotlib
matplotlib.use('Agg')
# 1. Importación de Librerías y Configuración
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configurar rutas
WORKSPACE_DIR = Path.cwd()
if WORKSPACE_DIR.name in ["notebooks", "02_gold_market_forecasting"]:
    BASE_DIR = WORKSPACE_DIR.parents[1] if WORKSPACE_DIR.name == "02_gold_market_forecasting" else WORKSPACE_DIR.parent
else:
    BASE_DIR = WORKSPACE_DIR

GOLD_FEATURES_FILE = BASE_DIR / "data" / "gold" / "features" / "features_market_forecasting.parquet"
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_theme(style="whitegrid")
print(f"Cargando features de mercado desde: {GOLD_FEATURES_FILE}")


Cargando features de mercado desde: C:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\Agrostat_app\data\gold\features\features_market_forecasting.parquet


## 2. Carga y Filtrado de Series Temporales desde el Feature Store
Seleccionamos una serie representativa para modelado: **Aguacate Hass (CPC 01211)** en **Corabastos Bogotá**.


In [2]:
# Carga del Feature Store Parquet
df_market = pd.read_parquet(GOLD_FEATURES_FILE)
df_market["fecha_completa"] = pd.to_datetime(df_market["fecha_completa"])

# Filtrar para un producto y mercado específico
df_series = df_market[
    (df_market["codigo_cpc"] == "01211") & 
    (df_market["mercado_id"] == "CORABASTOS")
].sort_values("fecha_completa").reset_index(drop=True)

df_series.set_index("fecha_completa", inplace=True)
print(f"Serie seleccionada: {len(df_series)} días consecutivos (Desde {df_series.index.min().date()} hasta {df_series.index.max().date()})")
df_series[["precio_promedio", "volumen_total_kg", "rolling_mean_7d", "precipitacion_mm"]].head()


Serie seleccionada: 180 días consecutivos (Desde 2026-01-01 hasta 2026-06-29)


,precio_promedio,volumen_total_kg,rolling_mean_7d,precipitacion_mm
fecha_completa,,,,
2026-01-01,4564.37,461897.0,4564.370000,5.5
2026-01-02,4783.02,284322.7,4673.695000,12.1
2026-01-03,4552.23,390198.9,4633.206667,13.9
2026-01-04,4513.30,408489.1,4603.230000,28.1
2026-01-05,4665.01,444887.0,4615.586000,11.8


## 3. Descomposición de la Serie y Prueba de Estacionariedad
Analizamos la estructura estocástica de la serie de precios promedio.


In [3]:
# 3.1 Descomposición Estacional (Período Semanal = 7 días)
decomp = seasonal_decompose(df_series["precio_promedio"], model="additive", period=7)
fig = decomp.plot()
fig.set_size_inches(12, 8)
plt.suptitle("Descomposición Aditiva de la Serie de Precios Mayoristas (Aguacate Hass - Corabastos)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# 3.2 Prueba de Dickey-Fuller Aumentada (ADF Test)
adf_res = adfuller(df_series["precio_promedio"].dropna())
print("--- Resultados del Test Aumentado de Dickey-Fuller (ADF) ---")
print(f"Estadístico ADF: {adf_res[0]:.4f}")
print(f"p-valor: {adf_res[1]:.5f}")
print(f"Valores Críticos: {adf_res[4]}")
if adf_res[1] <= 0.05:
    print(">> Conclusión: La serie es ESTACIONARIA (rechaza H0 con p <= 0.05).")
else:
    print(">> Conclusión: La serie NO es estacionaria; requiere diferenciación d=1.")


--- Resultados del Test Aumentado de Dickey-Fuller (ADF) ---
Estadístico ADF: -1.5295
p-valor: 0.51886
Valores Críticos: {'1%': np.float64(-3.470616369591229), '5%': np.float64(-2.8792214018977655), '10%': np.float64(-2.57619681359045)}
>> Conclusión: La serie NO es estacionaria; requiere diferenciación d=1.


## 4. División de Datos Temporal (Train / Test Split)
Para respetar la causalidad temporal y evitar fuga de información (*data leakage*), usamos los primeros 80% de días para entrenamiento y el último 20% para evaluación fuera de muestra (*out-of-sample*).


In [4]:
# Split cronológico sin shuffle
split_idx = int(len(df_series) * 0.85)
train_df = df_series.iloc[:split_idx]
test_df = df_series.iloc[split_idx:]

y_train = train_df["precio_promedio"]
y_test = test_df["precio_promedio"]
print(f"Período Train: {train_df.index.min().date()} a {train_df.index.max().date()} ({len(train_df)} observaciones)")
print(f"Período Test: {test_df.index.min().date()} a {test_df.index.max().date()} ({len(test_df)} observaciones)")


Período Train: 2026-01-01 a 2026-06-02 (153 observaciones)
Período Test: 2026-06-03 a 2026-06-29 (27 observaciones)


## 5. Modelado: Holt-Winters, SARIMAX y Random Forest Regressor
Ajustamos y evaluamos los tres enfoques de pronóstico.


In [5]:
# 5.1 Modelo 1: Suavizamiento Exponencial Holt-Winters
hw_model = ExponentialSmoothing(
    y_train,
    trend="add",
    seasonal="add",
    seasonal_periods=7,
    initialization_method="estimated"
).fit()
pred_hw = hw_model.forecast(len(test_df))

# 5.2 Modelo 2: SARIMAX con Covariable Climática Exógena (Precipitación)
exog_train = train_df[["precipitacion_mm", "temperatura_celsius"]]
exog_test = test_df[["precipitacion_mm", "temperatura_celsius"]]

sarimax_model = SARIMAX(
    y_train,
    exog=exog_train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)
pred_sarimax = sarimax_model.forecast(len(test_df), exog=exog_test)

# 5.3 Modelo 3: Random Forest Regressor basado en Lags y Ventanas Móviles
feature_cols = [
    "precio_lag_1d", "precio_lag_7d", "precio_lag_14d",
    "rolling_mean_7d", "rolling_std_7d", "precipitacion_mm", "dia_semana"
]
X_train_rf = train_df[feature_cols].bfill().ffill().fillna(0.0)
X_test_rf = test_df[feature_cols].bfill().ffill().fillna(0.0)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train_rf, y_train)
pred_rf = pd.Series(rf_model.predict(X_test_rf), index=test_df.index)

print("[OK] Todos los modelos ajustados y evaluados sobre el conjunto Test.")


C:\Users\ADAN\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\ADAN\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\Users\ADAN\AppData\Roaming\Python\Python313\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


[OK] Todos los modelos ajustados y evaluados sobre el conjunto Test.


C:\Users\ADAN\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 6. Comparación de Métricas y Visualización del Pronóstico
Calculamos $MAE, RMSE, MAPE$ y graficamos la trayectoria proyectada con intervalo de confianza al 95%.


In [6]:
# Cálculo de Métricas de Error
def calc_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0
    r2 = r2_score(y_true, y_pred)
    return {"Modelo": name, "MAE ($)": round(mae, 2), "RMSE ($)": round(rmse, 2), "MAPE (%)": round(mape, 2), "R²": round(r2, 3)}

metrics_df = pd.DataFrame([
    calc_metrics(y_test, pred_hw, "Holt-Winters"),
    calc_metrics(y_test, pred_sarimax, "SARIMAX + Clima Exógeno"),
    calc_metrics(y_test, pred_rf, "Random Forest Lagged Regressor")
])
display(metrics_df)

# Gráfico Comparativo del Pronóstico
plt.figure(figsize=(14, 6))
plt.plot(train_df.index[-30:], train_df["precio_promedio"].tail(30), label="Histórico Reciente (Train)", color="black", alpha=0.6)
plt.plot(test_df.index, y_test, label="Precio Real (Test)", color="black", linewidth=2.5)
plt.plot(test_df.index, pred_hw, label="Holt-Winters", linestyle="--", color="orange")
plt.plot(test_df.index, pred_sarimax, label="SARIMAX (Clima Exógeno)", linestyle="-.", color="blue")
plt.plot(test_df.index, pred_rf, label="Random Forest ML", linestyle=":", color="green", linewidth=2)

# Banda de Confianza al 95% para SARIMAX
forecast_res = sarimax_model.get_forecast(len(test_df), exog=exog_test)
ci = forecast_res.conf_int(alpha=0.05)
plt.fill_between(test_df.index, ci.iloc[:, 0], ci.iloc[:, 1], color="blue", alpha=0.15, label="IC 95% SARIMAX")

plt.title("Pronóstico Fuera de Muestra de Precios Mayoristas (SIPSA Corabastos - Aguacate Hass)", fontsize=13)
plt.xlabel("Fecha", fontsize=11)
plt.ylabel("Precio ($ COP / kg)", fontsize=11)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


,Modelo,MAE ($),RMSE ($),MAPE (%),R²
0,Holt-Winters,419.16,435.44,10.12,-8.641
1,SARIMAX + Clima Exógeno,179.03,210.30,4.34,-1.249
2,Random Forest Lagged Regressor,322.02,349.15,7.83,-5.199
